# Notebook 04: Keep only microglia and label their states

**Goal.** Cluster cells, remove contaminating cell types and doublets using marker genes, then label each cluster as homeostatic, DAM, interferon, proliferating, and so on.

States matter: DAM cells express more lysosomal genes, so a change in the *mix* of states could look like a change in coupling. Labelling states lets later notebooks separate the two.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **normalize**: divides each cell's counts by its total and multiplies by 10,000, then takes log(1+x), so deeply and shallowly sequenced cells become comparable. **pca**: picks the 2,000 most variable genes and compresses them into 30 axes. **cluster**: groups cells into 8 clusters (k-means on the first 20 axes).

*You should see:* Each cohort: cells, 8 clusters.

In [ ]:
from mlc.dataset import Dataset
from mlc import proc, plots
mg = {}
for c in cohorts()["mouse"]:
    ds = proc.normalize(Dataset.load(PROC / f"{c}_qc"))
    proc.pca(ds, n_comps=30); proc.cluster(ds, k=8, seed=SEED)
    mg[c] = ds; print(c, ds.n_obs, "cells,", ds.obs.cluster.nunique(), "clusters")

**Cell 3.** Scores every cell for **microglia identity** (P2ry12, Tmem119, Cx3cr1 ...) and **non-microglia identity** (neurons, oligodendrocytes, astrocytes, endothelium, lymphocytes). A score is the average of the gene set minus the average of expression-matched control genes, so 0 means 'nothing special'. The table lists each cluster's mean scores; **contaminant = True** when the non-microglia score beats the microglia score.

*You should see:* Mostly False; one or two small clusters are True (contaminants; scores about -1.9 and +1.3).

In [ ]:
for c, ds in mg.items():
    proc.add_scores(ds, {"micro": G.MICROGLIA, "nonmicro": G.NON_MICROGLIA})
    print(c); print(proc.cluster_table(ds))

**Cell 4.** Removes (1) whole contaminant clusters and (2) individual cells that express non-microglia genes unusually strongly (more than 5 robust standard deviations above the median), which also catches most doublets. This is a simple marker-based doublet filter; if you also want Scrublet, run it outside this guide and drop its flagged cells here.

*You should see:* One or two small clusters dropped plus about 150-200 flagged cells per cohort (about 8% of cells removed in total).

In [ ]:
for c, ds in mg.items():
    t = proc.cluster_table(ds)
    drop = list(t.index[t.contaminant])                     # EDIT to override, e.g. drop = ["3"]
    nm = ds.obs.nonmicro; thr = nm.median() + 5 * (nm - nm.median()).abs().median() * 1.4826
    flagged = (nm > thr).values                              # cells with non-microglia genes switched on
    keep = ~ds.obs.cluster.isin(drop).values & ~flagged
    print(c, "dropped clusters:", drop, "| flagged cells:", int(flagged.sum()), "| kept", int(keep.sum()), "of", ds.n_obs)
    mg[c] = ds.subset_cells(keep)

**Cell 5.** With contaminants gone, the cells are re-clustered (the clusters are used only for display). Each cell also receives a **dissociation-stress score** (Fos, Jun, Hspa1a ...; genes switched on by tissue dissociation, not by biology) and a score for every state's marker panel. A cell is called **DAM, interferon, proliferating, chemokine or MHC-II** if its best activated panel scores above **thr = 1.0** (about one log unit above matched controls); otherwise it is **homeostatic**. Raise thr for stricter calls; real data are noisier, so try 0.6 to 1.2 and check the heat map.

*You should see:* e.g. {'homeostatic': 3258, 'DAM': 820, 'interferon': 250, 'proliferating': 231, 'MHC_II': 92, 'chemokine_inflam': 26}. Rare states have few cells and a few are false calls (noisy 4-gene panels), which is why later notebooks analyse only the large states.

In [ ]:
for c, ds in mg.items():
    proc.normalize(ds); proc.pca(ds, n_comps=30); proc.cluster(ds, k=6, seed=SEED)
    ds.obs["dissoc"], _ = proc.score_genes(ds, G.DISSOCIATION)
    proc.add_scores(ds, G.STATE_PANELS, prefix="st_")
    Z = proc.label_states(ds, thr=1.0)
    print(c, ds.obs.state.value_counts().to_dict())

**Cell 6.** Left: identity scores per cluster (all clusters should now be microglia). Right: mean score of every marker panel (columns) in every called state (rows): the diagonal should be bright. In the dry run the table compares our calls with the simulated truth; nearly every simulated DAM cell should be called DAM.

In [ ]:
c0 = cohorts()["mouse"][0]
plots.fig_clusters(Z, proc.cluster_table(mg[c0])); plt.show()
if "true_state" in mg[c0].obs:       # simulated data only: how well did we recover the truth?
    print(pd.crosstab(mg[c0].obs.state, mg[c0].obs.true_state))

**Cell 7.** Saves the clean microglia with their state labels.

In [ ]:
for c, ds in mg.items():
    ds.save(PROC / f"{c}_microglia"); print(c, ds.obs.state.value_counts().to_dict())